# Episode 10: The Hub

Groups 1–2 gave you one agent with a rich harness. **Group 3 is the real thing: many autonomous agents.** And in AG2 Beta, autonomous agents don't call each other like functions — they communicate over a **Hub**.

**In this episode you'll build:** a Hub, register two agents on it, and have them exchange a message over a channel.

This is the foundation for the next five episodes.

## Delegation vs networking

In Episode 4 a coordinator *called* a sub-agent like a tool. That's **delegation** — one agent drives.

The Hub is different. Each agent is an autonomous, registered participant. They send each other **envelopes** through a central Hub, which:

- holds the **registry** of who's on the network
- keeps an **audit log** of everything that happens
- keeps a **write-ahead log (WAL)** per channel — a complete, replayable record

Every message goes through the Hub, so every interaction is observable and replayable.

## The pieces

| Piece | Role |
|---|---|
| `Hub` | One per network. The authoritative state — registry, audit log, WALs. |
| `LocalLink` | The in-process transport connecting clients to the Hub. |
| `HubClient` | One per agent. The broker that registers an agent and carries its traffic. |
| `Passport` | An agent's stable identity (`name`, a hub-stamped `agent_id`). |
| `Resume` | An agent's capability claims. |
| `Channel` | A bounded exchange between participants, governed by an *adapter*. |

You boot a `Hub`, wrap each `Agent` with `HubClient.register(...)`, then open a `Channel` to let them talk.

## Setup

Network code lives in `autogen.beta.network`. Importing it is what opts you into the network — a bare `Agent` still works standalone.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.knowledge import MemoryKnowledgeStore
from autogen.beta.network import (
    EV_CHANNEL_CLOSED,
    EV_TEXT,
    Hub,
    HubClient,
    LocalLink,
    Passport,
    Resume,
)

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

## The smallest network

This one cell does the whole thing: boot the Hub, register `alice` and `bob`, open a `consulting` channel (the simplest kind — one question, one answer), and replay the exchange from the WAL.

`attach_plugin=False` keeps the agents bare — they only need to send and reply here.

In [1]:
# 1. Boot the hub -- it holds the registry, audit log, and a WAL per channel.
hub = await Hub.open(MemoryKnowledgeStore(), ttl_sweep_interval=0)
link = LocalLink(hub)  # in-process transport

# 2. One HubClient per agent; register each Agent with a Passport + Resume.
alice_hc = HubClient(link, hub=hub)
bob_hc = HubClient(link, hub=hub)
alice = await alice_hc.register(
    Agent("alice", prompt="Ask one focused question and stop.", config=config),
    Passport(name="alice"),
    Resume(),
    attach_plugin=False,
)
bob = await bob_hc.register(
    Agent("bob", prompt="Answer in one short, clear sentence.", config=config),
    Passport(name="bob"),
    Resume(),
    attach_plugin=False,
)
print(f"hub running -- registered: {len(await hub.list_agents())} agents")

# 3. Agents talk over a channel, never by direct calls. Open one and send.
channel = await alice.open(type="consulting", target="bob")
await channel.send(
    "In one sentence: what is a write-ahead log?", audience=[bob.agent_id]
)

# 4. Wait for the hub to close the channel, then replay it from the WAL.
close_env = await alice.wait_for_channel_event(
    channel_id=channel.channel_id,
    predicate=lambda e: e.event_type == EV_CHANNEL_CLOSED,
    timeout=60.0,
)
print(f"channel closed: {close_env.event_data.get('reason')!r}")
print("--- WAL replay ---")
names = {alice.agent_id: "alice", bob.agent_id: "bob"}
for env in await hub.read_wal(channel.channel_id):
    if env.event_type == EV_TEXT:
        print(f"{names[env.sender_id]}: {env.event_data['text']}")

await alice_hc.close()
await bob_hc.close()
await hub.close()

hub running -- registered: 2 agents
channel closed: 'consulting_complete'
--- WAL replay ---
alice: In one sentence: what is a write-ahead log?
bob: A write-ahead log is a data recording method where changes are first logged to a durable log before being applied to the main database to ensure data integrity and support recovery.


## What just happened

You didn't wire `alice` to `bob`. You registered both on the Hub and opened a **channel**. The Hub routed `alice`'s question to `bob`, `bob`'s default handler ran his `Agent`, and the reply flowed back — all recorded on the WAL.

`hub.read_wal(channel_id)` is the payoff: a complete, replayable transcript of everything that happened, available to anyone who can see the channel.

## Additional content

**The `agent_id`.** The Hub stamps a unique `agent_id` at registration — use it for routing, not the human-readable name (`open(target="bob")` accepts the name as a convenience).

**`Hub.open` arguments.** `MemoryKnowledgeStore()` keeps everything in memory; swap in `DiskKnowledgeStore(path)` for durability across restarts. `ttl_sweep_interval=0` disables the time-to-live sweeper — fine for short demos.

**Always close.** Pair `Hub.open(...)` with `hub.close()`, and close each `HubClient`. In real code, use `try/finally`.

**Channels have adapters.** The `consulting` channel here is one of four adapter types. Each defines who can speak, in what order, and when the channel closes — the next four episodes take one each.

## What's next

You used a `consulting` channel without dwelling on it. Episode 11 makes it the subject — the strict one-question / one-answer adapter, and why its guarantees matter.